# AIS Analysis & Vessel Compatibility Experiments

This notebook develops the AIS-based vessel attribution pipeline for ORCA.

## Objective

Given:
- an observed oil-spill event,
- a reconstructed probable origin zone,
- an estimated spill time window,
- and AIS vessel trajectories,

the pipeline will:

1. Clean and validate AIS observations
2. Reconstruct vessel trajectories
3. Generate candidate vessels using spatial and temporal filtering
4. Extract vessel-behavior features
5. Train an XGBoost compatibility model
6. Produce compatibility probabilities
7. Rank candidate vessels
8. Compare ML compatibility with the existing evidence-based scoring system

> **Important:** The AIS data used initially in this notebook is synthetic.
> Synthetic ground-truth labels are used only to validate the ML pipeline.
> They are NOT real vessel-attribution labels and must not be presented as such.

## Pipeline

The experimental pipeline is:

AIS observations
        ↓
Data validation & cleaning
        ↓
Trajectory reconstruction
        ↓
Spatial / temporal candidate filtering
        ↓
Feature engineering
        ↓
Vessel-event feature matrix
        ↓
XGBoost compatibility model
        ↓
Compatibility probability
        ↓
Evidence fusion
        ↓
Final vessel ranking

### Core features

The feature set is aligned with the ORCA attribution logic:

- Minimum distance to probable origin
- Time difference from estimated spill event
- Time spent inside the origin region
- Trajectory consistency
- AIS quality
- Speed statistics
- Course behavior
- Distance travelled near the event
- Dwell / presence characteristics

In [ ]:
import math
import json
import random
from pathlib import Path
from datetime import datetime, timedelta

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

from sklearn.model_selection import train_test_split
from sklearn.metrics import (
    classification_report,
    confusion_matrix,
    roc_auc_score,
    accuracy_score,
)

try:
    from xgboost import XGBClassifier
except ImportError:
    XGBClassifier = None
    print("XGBoost is not installed in the current environment.")
    print("Install it later with: pip install xgboost")

print("Imports complete.")

In [ ]:
# Reproducibility
RANDOM_SEED = 42

np.random.seed(RANDOM_SEED)
random.seed(RANDOM_SEED)

# Synthetic experiment configuration
N_EVENTS = 40
VESSELS_PER_EVENT = 8
POSITIONS_PER_TRACK = 24

# AIS sampling interval
AIS_INTERVAL_HOURS = 1

# Candidate search radius
SEARCH_RADIUS_KM = 20.0

# Origin uncertainty radius
ORIGIN_RADIUS_KM = 10.0

print(f"Events: {N_EVENTS}")
print(f"Vessels per event: {VESSELS_PER_EVENT}")
print(f"Positions per trajectory: {POSITIONS_PER_TRACK}")
print(f"Search radius: {SEARCH_RADIUS_KM} km")

## 1. Define a Synthetic Spill Event

Each synthetic event contains:

- an observed spill location,
- an estimated origin,
- an event timestamp,
- a permissible attribution time window.

For the synthetic experiment, one vessel is deliberately generated as the
ground-truth associated vessel.

This gives us a controlled label for evaluating whether the ML pipeline
can learn compatibility patterns.

The labels represent **synthetic ground truth only**.

In [ ]:
def haversine_km(lat1, lon1, lat2, lon2):
    """
    Calculate great-circle distance between two geographic points.
    """
    R = 6371.0

    lat1 = np.radians(lat1)
    lat2 = np.radians(lat2)
    dlat = lat2 - lat1
    dlon = np.radians(lon2 - lon1)

    a = (
        np.sin(dlat / 2) ** 2
        + np.cos(lat1) * np.cos(lat2) * np.sin(dlon / 2) ** 2
    )

    return 2 * R * np.arcsin(np.sqrt(a))


def destination_point(lat, lon, distance_km, bearing_deg):
    """
    Move from a geographic point by distance_km along bearing_deg.
    """
    R = 6371.0

    lat_rad = np.radians(lat)
    lon_rad = np.radians(lon)
    bearing_rad = np.radians(bearing_deg)

    angular_distance = distance_km / R

    new_lat = np.arcsin(
        np.sin(lat_rad) * np.cos(angular_distance)
        + np.cos(lat_rad)
        * np.sin(angular_distance)
        * np.cos(bearing_rad)
    )

    new_lon = lon_rad + np.arctan2(
        np.sin(bearing_rad) * np.sin(angular_distance) * np.cos(lat_rad),
        np.cos(angular_distance) - np.sin(lat_rad) * np.sin(new_lat),
    )

    return np.degrees(new_lat), np.degrees(new_lon)


def create_event(event_id):
    """
    Generate one synthetic oil-spill event.
    """
    origin_lat = np.random.uniform(12.0, 16.0)
    origin_lon = np.random.uniform(72.0, 76.0)

    event_time = datetime(2026, 1, 1) + timedelta(
        hours=np.random.randint(0, 24 * 120)
    )

    return {
        "event_id": event_id,
        "origin_lat": origin_lat,
        "origin_lon": origin_lon,
        "event_time": event_time,
        "window_start": event_time - timedelta(hours=6),
        "window_end": event_time + timedelta(hours=6),
    }


events = pd.DataFrame(
    [create_event(i) for i in range(N_EVENTS)]
)

events.head()

## 2. Generate Synthetic AIS Trajectories

For each event we generate several vessels.

One vessel is designated as the synthetic ground-truth associated vessel.

### Responsible-vessel behavior

Its trajectory is generated so that it:

- approaches the probable origin,
- passes relatively close to it,
- is temporally aligned with the event,
- remains in the vicinity for some time.

### Distractor-vessel behavior

Other vessels are generated with varying:

- distances,
- speeds,
- headings,
- temporal offsets,
- trajectories.

This allows the model to learn vessel-event compatibility.

In [ ]:
def generate_vessel_track(
    event,
    vessel_index,
    responsible=False,
    n_positions=POSITIONS_PER_TRACK,
):
    """
    Generate one synthetic AIS trajectory for an event.
    """

    event_time = event["event_time"]

    vessel_id = f"{event['event_id']:03d}{vessel_index:03d}"

    if responsible:
        # Start farther away and move toward the origin.
        start_distance = np.random.uniform(15, 30)
        start_bearing = np.random.uniform(0, 360)

        start_lat, start_lon = destination_point(
            event["origin_lat"],
            event["origin_lon"],
            start_distance,
            start_bearing,
        )

        target_lat = event["origin_lat"]
        target_lon = event["origin_lon"]

    else:
        # Distractors begin at a broader range of positions.
        start_distance = np.random.uniform(10, 80)
        start_bearing = np.random.uniform(0, 360)

        start_lat, start_lon = destination_point(
            event["origin_lat"],
            event["origin_lon"],
            start_distance,
            start_bearing,
        )

        target_bearing = np.random.uniform(0, 360)
        target_distance = np.random.uniform(10, 80)

        target_lat, target_lon = destination_point(
            event["origin_lat"],
            event["origin_lon"],
            target_distance,
            target_bearing,
        )

    rows = []

    for i in range(n_positions):

        fraction = i / (n_positions - 1)

        # Small movement noise
        lat = (
            start_lat
            + fraction * (target_lat - start_lat)
            + np.random.normal(0, 0.003)
        )

        lon = (
            start_lon
            + fraction * (target_lon - start_lon)
            + np.random.normal(0, 0.003)
        )

        if responsible:
            # Force trajectory to pass near event time/origin.
            if 8 <= i <= 15:
                proximity_noise = np.random.normal(0, 0.004)

                lat = event["origin_lat"] + proximity_noise
                lon = event["origin_lon"] + proximity_noise

        speed = (
            np.random.uniform(7, 14)
            if responsible
            else np.random.uniform(4, 18)
        )

        course = np.random.uniform(0, 360)

        timestamp = event_time + timedelta(
            hours=(i - n_positions // 2) * AIS_INTERVAL_HOURS
        )

        rows.append(
            {
                "event_id": event["event_id"],
                "mmsi": vessel_id,
                "timestamp": timestamp,
                "latitude": lat,
                "longitude": lon,
                "speed_knots": speed,
                "course_deg": course,
                "heading_deg": course + np.random.normal(0, 3),
                "vessel_type": random.choice(
                    ["Tanker", "Cargo", "Container", "Other"]
                ),
                "responsible_ground_truth": int(responsible),
            }
        )

    return rows


ais_rows = []

for _, event in events.iterrows():

    responsible_index = np.random.randint(VESSELS_PER_EVENT)

    for vessel_index in range(VESSELS_PER_EVENT):

        is_responsible = vessel_index == responsible_index

        ais_rows.extend(
            generate_vessel_track(
                event,
                vessel_index,
                responsible=is_responsible,
            )
        )

ais_df = pd.DataFrame(ais_rows)

print("AIS observations:", len(ais_df))
print("Unique vessels:", ais_df["mmsi"].nunique())
print("Events:", ais_df["event_id"].nunique())

ais_df.head()

In [ ]:
# Basic AIS validation

ais_df["timestamp"] = pd.to_datetime(ais_df["timestamp"], errors="coerce")

ais_df = ais_df.dropna(
    subset=[
        "mmsi",
        "timestamp",
        "latitude",
        "longitude",
        "speed_knots",
        "course_deg",
    ]
).copy()

# Geographic validity
ais_df = ais_df[
    ais_df["latitude"].between(-90, 90)
    & ais_df["longitude"].between(-180, 180)
]

# Physical AIS constraints
ais_df = ais_df[
    ais_df["speed_knots"].between(0, 100)
    & ais_df["course_deg"].between(0, 360)
]

ais_df = ais_df.sort_values(
    ["event_id", "mmsi", "timestamp"]
).reset_index(drop=True)

print("Clean observations:", len(ais_df))
print("Missing values:")
print(ais_df.isna().sum())

## 3. Visualize a Vessel Trajectory

Before feature engineering, we inspect the raw AIS trajectory.

The trajectory is represented as a sequence of latitude/longitude
observations ordered by timestamp.

In [ ]:
example_event_id = 0

example_event = events[
    events["event_id"] == example_event_id
].iloc[0]

example_ais = ais_df[
    ais_df["event_id"] == example_event_id
]

plt.figure(figsize=(9, 7))

for mmsi, group in example_ais.groupby("mmsi"):
    plt.plot(
        group["longitude"],
        group["latitude"],
        marker="o",
        markersize=2,
        alpha=0.6,
    )

plt.scatter(
    example_event["origin_lon"],
    example_event["origin_lat"],
    marker="*",
    s=200,
    label="Probable origin",
)

plt.xlabel("Longitude")
plt.ylabel("Latitude")
plt.title("Synthetic AIS Vessel Trajectories")
plt.legend()
plt.grid(True)
plt.show()

## 4. Calculate Vessel-Event Spatial Features

For every vessel associated with an event we calculate:

### Minimum distance

\[
d_{min} = \min_i d(x_i, x_{origin})
\]

where \(d\) is the haversine distance.

### Time of closest approach

The timestamp corresponding to \(d_{min}\).

### Temporal difference

\[
\Delta t = |t_{closest} - t_{event}|
\]

These features form the first layer of vessel compatibility.

In [ ]:
def calculate_spatial_temporal_features(
    vessel_df,
    origin_lat,
    origin_lon,
    event_time,
    origin_radius_km=ORIGIN_RADIUS_KM,
):
    """
    Calculate spatial and temporal vessel-event features.
    """

    df = vessel_df.copy()

    df["distance_to_origin_km"] = haversine_km(
        df["latitude"].values,
        df["longitude"].values,
        origin_lat,
        origin_lon,
    )

    closest_idx = df["distance_to_origin_km"].idxmin()
    closest = df.loc[closest_idx]

    min_distance = float(
        closest["distance_to_origin_km"]
    )

    closest_time = closest["timestamp"]

    time_difference_hours = abs(
        (closest_time - event_time).total_seconds()
    ) / 3600.0

    inside_region = df[
        df["distance_to_origin_km"] <= origin_radius_km
    ]

    if len(inside_region) > 1:
        time_inside_hours = (
            inside_region["timestamp"].max()
            - inside_region["timestamp"].min()
        ).total_seconds() / 3600.0
    else:
        time_inside_hours = 0.0

    return {
        "min_distance_km": min_distance,
        "closest_approach_time": closest_time,
        "time_difference_hours": time_difference_hours,
        "time_inside_region_hours": time_inside_hours,
    }

## 5. Calculate Trajectory-Behavior Features

Spatial proximity alone is insufficient.

A vessel may simply pass near the spill origin.

Therefore we also characterize its trajectory.

Useful indicators include:

- speed statistics,
- speed variability,
- course variability,
- trajectory consistency,
- AIS observation density,
- dwell behavior.

These provide behavioral evidence that can complement the
spatial and temporal signals.

In [ ]:
def calculate_behavior_features(vessel_df):
    """
    Calculate trajectory and AIS-quality features.
    """

    df = vessel_df.sort_values("timestamp").copy()

    speeds = df["speed_knots"].to_numpy()
    courses = df["course_deg"].to_numpy()

    speed_mean = float(np.mean(speeds))
    speed_std = float(np.std(speeds))

    # Circular course difference
    course_diffs = np.abs(np.diff(courses))
    course_diffs = np.minimum(
        course_diffs,
        360 - course_diffs,
    )

    course_change_mean = (
        float(np.mean(course_diffs))
        if len(course_diffs) > 0
        else 0.0
    )

    # Observation density
    time_span_hours = (
        df["timestamp"].max() - df["timestamp"].min()
    ).total_seconds() / 3600.0

    expected_positions = max(
        1,
        int(round(time_span_hours / AIS_INTERVAL_HOURS)) + 1
    )

    ais_quality = min(
        1.0,
        len(df) / expected_positions
    )

    # Directional consistency.
    # Low mean course change = more consistent trajectory.
    trajectory_consistency = 1.0 / (
        1.0 + course_change_mean / 45.0
    )

    return {
        "speed_mean_knots": speed_mean,
        "speed_std_knots": speed_std,
        "course_change_mean_deg": course_change_mean,
        "trajectory_consistency": trajectory_consistency,
        "ais_quality": ais_quality,
        "positions_used": len(df),
    }

## 6. Build the Vessel-Event Feature Matrix

Each row now represents:

> **one vessel evaluated against one oil-spill event**

This is important because the XGBoost model should learn:

\[
P(\text{vessel compatible} \mid \text{event, AIS features})
\]

rather than simply learning characteristics of individual vessels.

In [ ]:
feature_rows = []

for _, event in events.iterrows():

    event_ais = ais_df[
        ais_df["event_id"] == event["event_id"]
    ]

    for mmsi, vessel_df in event_ais.groupby("mmsi"):

        spatial_features = calculate_spatial_temporal_features(
            vessel_df,
            event["origin_lat"],
            event["origin_lon"],
            event["event_time"],
        )

        behavior_features = calculate_behavior_features(
            vessel_df
        )

        row = {
            "event_id": event["event_id"],
            "mmsi": mmsi,
            **spatial_features,
            **behavior_features,
            "label": int(
                vessel_df["responsible_ground_truth"].iloc[0]
            ),
        }

        feature_rows.append(row)


vessel_features = pd.DataFrame(feature_rows)

print("Vessel-event samples:", len(vessel_features))
print("Positive samples:", vessel_features["label"].sum())
print("Negative samples:", (vessel_features["label"] == 0).sum())

vessel_features.head()

## 7. Candidate Filtering

Before ML ranking, the production pipeline should eliminate vessels that
are clearly irrelevant.

A basic candidate-generation rule is:

\[
d_{min} \leq R_{search}
\]

where \(R_{search}\) is the search radius around the reconstructed origin.

This is a **candidate-generation step**, not the final attribution decision.

The ML model operates on the remaining candidate vessels.

In [ ]:
candidate_df = vessel_features[
    vessel_features["min_distance_km"] <= SEARCH_RADIUS_KM
].copy()

print("Total vessel-event samples:", len(vessel_features))
print("Candidates after spatial filtering:", len(candidate_df))

candidate_df[
    [
        "event_id",
        "mmsi",
        "min_distance_km",
        "time_difference_hours",
        "time_inside_region_hours",
        "label",
    ]
].head(10)

## 8. Prepare Features for XGBoost

The model uses numerical evidence features.

We deliberately exclude:

- `event_id`
- `mmsi`
- timestamps
- the synthetic ground-truth label.

The model therefore cannot simply memorize a particular vessel or event.

### Feature set

- `min_distance_km`
- `time_difference_hours`
- `time_inside_region_hours`
- `trajectory_consistency`
- `ais_quality`
- `speed_mean_knots`
- `speed_std_knots`
- `course_change_mean_deg`
- `positions_used`

In [ ]:
FEATURE_COLUMNS = [
    "min_distance_km",
    "time_difference_hours",
    "time_inside_region_hours",
    "trajectory_consistency",
    "ais_quality",
    "speed_mean_knots",
    "speed_std_knots",
    "course_change_mean_deg",
    "positions_used",
]

X = vessel_features[FEATURE_COLUMNS].copy()
y = vessel_features["label"].copy()

print("Feature matrix shape:", X.shape)
print("Features:")
for feature in FEATURE_COLUMNS:
    print(" -", feature)

## 9. Train/Validation Split

AIS observations belonging to the same oil-spill event are correlated.

Therefore we should avoid randomly splitting individual vessel rows across
training and validation data.

Instead, events are split first.

This prevents vessels from the same synthetic event appearing in both
training and validation sets.

In [ ]:
event_ids = vessel_features["event_id"].unique()

train_events, val_events = train_test_split(
    event_ids,
    test_size=0.2,
    random_state=RANDOM_SEED,
)

train_mask = vessel_features["event_id"].isin(train_events)
val_mask = vessel_features["event_id"].isin(val_events)

X_train = vessel_features.loc[
    train_mask, FEATURE_COLUMNS
]

y_train = vessel_features.loc[
    train_mask, "label"
]

X_val = vessel_features.loc[
    val_mask, FEATURE_COLUMNS
]

y_val = vessel_features.loc[
    val_mask, "label"
]

print("Training events:", len(train_events))
print("Validation events:", len(val_events))
print("Training samples:", len(X_train))
print("Validation samples:", len(X_val))

In [ ]:
if XGBClassifier is not None:

    model = XGBClassifier(
        n_estimators=250,
        max_depth=4,
        learning_rate=0.05,
        subsample=0.8,
        colsample_bytree=0.8,
        objective="binary:logistic",
        eval_metric="logloss",
        random_state=RANDOM_SEED,
    )

    model.fit(
        X_train,
        y_train,
        eval_set=[(X_val, y_val)],
        verbose=False,
    )

    print("XGBoost training complete.")

else:
    model = None
    print("Skipping training because XGBoost is unavailable.")

## 10. Evaluate the Compatibility Model

The main output is a probability:

\[
P(y=1 \mid X)
\]

where:

- \(y=1\) = vessel is compatible with the synthetic event
- \(X\) = AIS-derived evidence features.

The probability is used as a **compatibility score**, not as legal proof
of responsibility.

In [ ]:
if model is not None:

    val_probabilities = model.predict_proba(X_val)[:, 1]
    val_predictions = (val_probabilities >= 0.5).astype(int)

    print("Accuracy:")
    print(accuracy_score(y_val, val_predictions))

    print("\nROC-AUC:")
    print(roc_auc_score(y_val, val_probabilities))

    print("\nClassification report:")
    print(
        classification_report(
            y_val,
            val_predictions,
            zero_division=0,
        )
    )

    print("\nConfusion matrix:")
    print(confusion_matrix(y_val, val_predictions))

## 11. Feature Importance

Feature importance provides an initial interpretation of which evidence
signals the model is using.

This is useful for checking whether the learned model is relying on
reasonable attribution evidence rather than irrelevant identifiers.

In [ ]:
if model is not None:

    importance = pd.DataFrame(
        {
            "feature": FEATURE_COLUMNS,
            "importance": model.feature_importances_,
        }
    ).sort_values(
        "importance",
        ascending=False,
    )

    display(importance)

    plt.figure(figsize=(9, 6))

    plt.barh(
        importance["feature"],
        importance["importance"],
    )

    plt.xlabel("Importance")
    plt.ylabel("Feature")
    plt.title("XGBoost Vessel Compatibility Feature Importance")
    plt.gca().invert_yaxis()
    plt.grid(True, axis="x")
    plt.show()

## 12. Generate Compatibility Scores

The trained model produces a compatibility probability for every
vessel-event pair.

This probability can later be combined with the deterministic evidence
scores already implemented in the ORCA backend.

For example:

\[
S_{final}
=
w_{ML}S_{ML}
+
w_{spatial}S_{spatial}
+
w_{temporal}S_{temporal}
+
w_{trajectory}S_{trajectory}
+
w_{dwell}S_{dwell}
+
w_{AIS}S_{AIS}
\]

The exact production weights should be calibrated using real labelled
historical spill events.

In [ ]:
if model is not None:

    vessel_features["ml_compatibility"] = model.predict_proba(
        vessel_features[FEATURE_COLUMNS]
    )[:, 1]

else:

    vessel_features["ml_compatibility"] = np.nan

vessel_features[
    [
        "event_id",
        "mmsi",
        "min_distance_km",
        "time_difference_hours",
        "ml_compatibility",
        "label",
    ]
].head(10)

## 13. Rank Vessels for an Individual Event

For a selected spill event, rank candidate vessels by the learned
compatibility probability.

Higher probability means:

> the vessel's observed AIS behavior is more compatible with the
> characteristics learned from the training examples.

It does **not** mean the vessel is proven responsible.

In [ ]:
selected_event = 0

ranking = vessel_features[
    vessel_features["event_id"] == selected_event
].copy()

ranking = ranking.sort_values(
    "ml_compatibility",
    ascending=False,
).reset_index(drop=True)

ranking["rank"] = np.arange(1, len(ranking) + 1)

ranking[
    [
        "rank",
        "mmsi",
        "ml_compatibility",
        "min_distance_km",
        "time_difference_hours",
        "time_inside_region_hours",
        "trajectory_consistency",
        "label",
    ]
]

## 14. Inspect the Ground-Truth Vessel

For this synthetic experiment only, we can verify where the ground-truth
vessel appears in the ranking.

This check would NOT exist in a real deployment because the responsible
vessel is generally unknown at prediction time.

In [ ]:
ground_truth_rows = ranking[
    ranking["label"] == 1
]

if len(ground_truth_rows) > 0:

    gt_rank = int(
        ground_truth_rows["rank"].iloc[0]
    )

    gt_probability = float(
        ground_truth_rows["ml_compatibility"].iloc[0]
    )

    print("Synthetic ground-truth vessel:")
    print(ground_truth_rows["mmsi"].iloc[0])

    print("Rank:", gt_rank)
    print("ML compatibility:", round(gt_probability, 4))

## 15. Compare ML Compatibility With Deterministic Evidence

The current ORCA attribution pipeline already uses evidence signals such as:

- spatial proximity,
- temporal alignment,
- trajectory consistency,
- dwell time,
- AIS quality.

The ML model should therefore be treated as an additional compatibility
layer rather than automatically replacing the deterministic scorer.

This allows the system to preserve interpretable evidence while gradually
introducing learned relationships.

In [ ]:
def normalize_inverse(value, scale):
    """
    Convert a positive distance/time quantity into a [0, 1] compatibility
    score where smaller values are better.
    """
    return float(np.exp(-value / scale))


ranking["spatial_score"] = ranking[
    "min_distance_km"
].apply(
    lambda x: normalize_inverse(x, SEARCH_RADIUS_KM)
)

ranking["temporal_score"] = ranking[
    "time_difference_hours"
].apply(
    lambda x: normalize_inverse(x, 12.0)
)

ranking["dwell_score"] = np.clip(
    ranking["time_inside_region_hours"] / 6.0,
    0,
    1,
)

ranking["trajectory_score"] = ranking[
    "trajectory_consistency"
]

ranking["ais_quality_score"] = ranking[
    "ais_quality"
]

# Example experimental fusion weights.
# These are NOT calibrated production weights.
ranking["evidence_score"] = (
    0.35 * ranking["spatial_score"]
    + 0.25 * ranking["temporal_score"]
    + 0.20 * ranking["trajectory_score"]
    + 0.10 * ranking["dwell_score"]
    + 0.10 * ranking["ais_quality_score"]
)

ranking["fused_score"] = (
    0.60 * ranking["evidence_score"]
    + 0.40 * ranking["ml_compatibility"]
)

ranking = ranking.sort_values(
    "fused_score",
    ascending=False,
).reset_index(drop=True)

ranking["final_rank"] = np.arange(
    1,
    len(ranking) + 1
)

ranking[
    [
        "final_rank",
        "mmsi",
        "ml_compatibility",
        "evidence_score",
        "fused_score",
        "min_distance_km",
        "time_difference_hours",
        "label",
    ]
]

## 16. Save the Experimental Model

The trained model is exported separately from the application code.

Production inference will eventually load this artifact instead of
containing training logic inside `app/`.

Target artifact:

`../models/vessel_compatibility/xgb_vessel.json`

This model is currently trained on synthetic data and must NOT be treated
as a production attribution model.

In [ ]:
model_dir = Path("../models/vessel_compatibility")
model_dir.mkdir(
    parents=True,
    exist_ok=True,
)

model_path = model_dir / "xgb_vessel.json"

if model is not None:

    model.save_model(model_path)

    print("Model saved to:")
    print(model_path.resolve())

else:
    print("Model was not saved because XGBoost is unavailable.")

# Experimental Conclusions

The notebook now demonstrates the complete AIS intelligence pipeline:

1. Synthetic AIS generation
2. AIS validation
3. Vessel trajectory reconstruction
4. Spatial feature extraction
5. Temporal feature extraction
6. Behavioral feature extraction
7. Candidate filtering
8. Vessel-event feature matrix construction
9. Event-level train/validation split
10. XGBoost compatibility training
11. Model evaluation
12. Feature importance analysis
13. Compatibility probability generation
14. Vessel ranking
15. Evidence + ML score fusion
16. Model export

## Important limitation

The current model is trained entirely on synthetic data.

It demonstrates the **technical ML pipeline**, not real-world attribution
accuracy.

For the final ORCA system, the model should be retrained using historical
oil-spill events with:

- satellite-derived spill observations,
- estimated spill origin/time,
- corresponding AIS tracks,
- and reliable vessel-event attribution labels.

The existing deterministic evidence scorer can remain the fallback ranking
mechanism until such labelled data is available.